In [0]:
# Initial Load
passengers_day1 = [ 
(101,"Rahul Sharma","Hyderabad","Economy","India"), 
(102,"Priya Reddy","Bangalore","Business","India"), 
(103,"Amit Kumar","Mumbai","Economy","India"), 
(104,"Sneha Patel","Delhi","Premium Economy","India"), 
(105,"Farhan Ali","Chennai","Economy","India") 
]

columns = [ 
"passenger_id", 
"passenger_name", 
"city", 
"travel_class", 
"country" 
]

df_day1 = spark.createDataFrame( 
    passengers_day1,
    columns
)

In [0]:
# Incremental Load
passengers_day2 = [ 
(102,"Priya Reddy","Bangalore","First Class","India"), 
(104,"Sneha Patel","Hyderabad","Premium Economy","India"), 
(106,"Neha Singh","Pune","Economy","India"), 
(107,"Arjun Verma","Kochi","Business","India") 
]

df_day2 = spark.createDataFrame( 
    passengers_day2, 
    columns 
)

### Exercises
Create Delta

In [0]:
df_day1.write.format("delta").mode("overwrite").saveAsTable("passengers")

In [0]:
cnt_day_1 = spark.table("passengers").count()
print("Total records in day 1:", cnt_day_1)

Total records in day 1: 5


In [0]:
spark.sql("select * from passengers").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
%sql
DESCRIBE HISTORY passengers

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T10:58:27.000Z,144530056129212,azuser7221_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3535285718984215),7b363069-19da-4287-a1d9-a9b071b53559,0617-093035-o1gkbrp2-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2005)",null,Databricks-Runtime/18.2.x-photon-scala2.13


Merge / Upsert

In [0]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "passengers")
deltaTable.alias("target").merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
print("Verifying Passenger 102:")
display(spark.sql("SELECT * from passengers where passenger_id = 102"))

Verifying Passenger 102:


passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,First Class,India


In [0]:
print("Verifying Passenger 106:")
display(spark.sql("SELECT * from passengers where passenger_id = 106"))

Verifying Passenger 106:


passenger_id,passenger_name,city,travel_class,country
106,Neha Singh,Pune,Economy,India


Time Travel

In [0]:
print("Version 0:")
dfv0 = spark.read.format("delta").option("versionAsOf", 0).table("passengers")
display(dfv0)

print("Latest Version:")
df_latest = spark.read.format("delta").table("passengers")
display(df_latest)

print("Version 0 and Latest Version:")
display(dfv0.subtract(df_latest))

print("Passenger 102 in Version 0 and Latest Version:")
display(spark.sql("""SELECT 'Version 0' AS version, * FROM passengers VERSION AS OF 0 WHERE passenger_id = 102 UNION ALL SELECT 'Latest Version' AS version, * FROM passengers WHERE passenger_id = 102"""))

print("Passenger 104 in Version 0 and Latest Version:")
display(spark.sql("""SELECT 'Version 0' AS version, * FROM passengers VERSION AS OF 0 WHERE passenger_id = 104 UNION ALL SELECT 'Latest Version' AS version, * FROM passengers WHERE passenger_id = 104"""))

Version 0:


passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


Latest Version:


passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


Version 0 and Latest Version:


passenger_id,passenger_name,city,travel_class,country
104,Sneha Patel,Delhi,Premium Economy,India
102,Priya Reddy,Bangalore,Business,India


Passenger 102 in Version 0 and Latest Version:


version,passenger_id,passenger_name,city,travel_class,country
Version 0,102,Priya Reddy,Bangalore,Business,India
Latest Version,102,Priya Reddy,Bangalore,First Class,India


Passenger 104 in Version 0 and Latest Version:


version,passenger_id,passenger_name,city,travel_class,country
Version 0,104,Sneha Patel,Delhi,Premium Economy,India
Latest Version,104,Sneha Patel,Hyderabad,Premium Economy,India


Maintenance

In [0]:
print("Optimize:")
display(spark.sql("OPTIMIZE passengers"))

print("Z-Order:")
display(spark.sql("OPTIMIZE passengers ZORDER BY (city)"))

display(spark.sql("DELETE FROM passengers WHERE passenger_id = 105"))

print("History:")
display(spark.sql("DESCRIBE HISTORY passengers"))

print("Vacuum:")
display(spark.sql("VACUUM passengers"))

Optimize:


path,metrics
abfss://unity-catalog-storage@dbstoragequsa5urli33bk.dfs.core.windows.net/7405608869077142/__unitystorage/catalogs/96218beb-5f88-46b5-9915-219047b0f534/tables/eb18ac86-dd89-472a-8812-4b2af6f1f827,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781695864322, 1781695864854, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


Z-Order:


path,metrics
abfss://unity-catalog-storage@dbstoragequsa5urli33bk.dfs.core.windows.net/7405608869077142/__unitystorage/catalogs/96218beb-5f88-46b5-9915-219047b0f534/tables/eb18ac86-dd89-472a-8812-4b2af6f1f827,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2056), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781695865772, 1781695866166, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


num_affected_rows
1


History:


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-06-17T11:31:08.000Z,144530056129212,azuser7221_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#15624L = 105)""])",null,List(3535285718984215),a4376e1c-5a1a-4eba-b4b4-b31c2f56f555,0617-093035-o1gkbrp2-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1193, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 845, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 343)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-17T11:07:33.000Z,144530056129212,azuser7221_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3535285718984215),1d0195d8-7456-4662-b188-d494e0ac0da9,0617-093035-o1gkbrp2-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9128, p25FileSize -> 2056, numDeletionVectorsRemoved -> 1, minFileSize -> 2056, numAddedFiles -> 1, maxFileSize -> 2056, p75FileSize -> 2056, p50FileSize -> 2056, numAddedBytes -> 2056)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-17T11:07:32.000Z,144530056129212,azuser7221_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#14598L = passenger_id#14623L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3535285718984215),1d0195d8-7456-4662-b188-d494e0ac0da9,0617-093035-o1gkbrp2-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7123, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2927, materializeSourceTimeMs -> 81, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 979, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1843)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-17T10:58:27.000Z,144530056129212,azuser7221_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3535285718984215),7b363069-19da-4287-a1d9-a9b071b53559,0617-093035-o1gkbrp2-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2005)",null,Databricks-Runtime/18.2.x-photon-scala2.13


Vacuum:


path
abfss://unity-catalog-storage@dbstoragequsa5urli33bk.dfs.core.windows.net/7405608869077142/__unitystorage/catalogs/96218beb-5f88-46b5-9915-219047b0f534/tables/eb18ac86-dd89-472a-8812-4b2af6f1f827
